# 03 Initial Baselines

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`common.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/common.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)
- [`extra_trees.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/extra_trees.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)
- [`hgb.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/hgb.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)
- [`knn.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/knn.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)
- [`mlp.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/mlp.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)
- [`pixel_transform.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/pixel_transform.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)
- [`run.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/run.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)
- [`run_experiment2.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/run_experiment2.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)
- [`svc.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/svc.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'03_experiments/model_optimization/common.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '5e3fe87297cc69543778813449244cef7f33502e'}, '03_experiments/model_optimization/extra_trees.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '688200c338093c4609ec566f3d9ce4620835a594'}, '03_experiments/model_optimization/hgb.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'a63b379ad3f7a457bae4d5f459a21bff8480d331'}, '03_experiments/model_optimization/knn.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '0f2667be1124943ad81751ecd0ced68fd9b76720'}, '03_experiments/model_optimization/mlp.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '8377932f520d1efd9560856d9e3e1f3ce8e68cb8'}, '03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/pixel_transform.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '2878984820965b7c8b1478dc6797069eed41de4b'}, '03_experiments/model_optimization/run.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '53dd28043a7250a1fa0e935e4945d16e42302b98'}, '03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/run_experiment2.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'f3895529c8715f1292ce89bbe9a991a25c175524'}, '03_experiments/model_optimization/svc.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'c4af494a2b9d6e2367c3d93d1b9fccc88ac7b6c3'}}

## common

원본: [03_experiments/model_optimization/common.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/common.py)  
Git blob: `5e3fe87297cc69543778813449244cef7f33502e`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

Exact dataset partition and scaler matching W02 EXP-유-003.



In [ ]:
%%archive_source 03_experiments/model_optimization/common.py
"""Exact dataset partition and scaler matching W02 EXP-유-003."""
from __future__ import annotations
import hashlib
import sys
from pathlib import Path
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_recall_fscore_support

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "W02_2026-11-09_11-15" / "scripts" / "experiment2_baseline"))
from pixel_transform import PixelScaler255

DEFAULT_DATA = Path(__file__).resolve().parents[2] / "02_data" / "datasets" / "MNIST_EMNIST_official_split_112000.npz"
EXPECTED_SHA = "0e9570d0eb04c0c1da99d043b85bfdab7f3b639eebe4b10335b05b57ad671879"
LABELS = "0123456789ABCDEF"

def fixed_split(path=DEFAULT_DATA):
    """Official X_train only; replicate baseline's first StratifiedGroupKFold."""
    path = Path(path)
    with path.open("rb") as f:
        sha = hashlib.file_digest(f, "sha256").hexdigest()
    if sha != EXPECTED_SHA:
        raise ValueError("Dataset SHA-256 mismatch: " + sha)
    with np.load(path, allow_pickle=False) as d:
        X = d["X_train"]
        y = d["y_train"].astype(np.int64)
        src = d["train_source_index"].astype(np.int64)
        split = d["train_source_split"].astype(np.int64)
        aug = d["train_is_augmented"]
    if X.shape != (96000, 28, 28) or int(aug.sum()) != 5349:
        raise ValueError("Unexpected data/augmentation count")
    groups = y * 1_000_000_000 + split * 100_000_000 + src
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    tr, va = next(cv.split(X, y, groups))
    if (len(tr), len(va)) != (76800, 19200):
        raise ValueError("Baseline split sizes not reproduced")
    if np.intersect1d(groups[tr], groups[va]).size:
        raise ValueError("Leakage across original/augmentation groups")
    return (X[tr].reshape(-1, 784).astype(np.float32), y[tr],
            X[va].reshape(-1, 784).astype(np.float32), y[va], sha)

def pipeline(model, scaled):
    return Pipeline([("scale_0_1", PixelScaler255()), ("model", model)]) if scaled else model

def scores(y_true, y_pred):
    ids = np.arange(16)
    p, r, f, n = precision_recall_fscore_support(y_true, y_pred, labels=ids, zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=ids, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=ids).tolist(),
        "per_class": {c: {"precision":float(p[i]),"recall":float(r[i]),"f1":float(f[i]),"support":int(n[i])} for i,c in enumerate(LABELS)}
    }


## extra_trees

원본: [03_experiments/model_optimization/extra_trees.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/extra_trees.py)  
Git blob: `688200c338093c4609ec566f3d9ce4620835a594`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

Extra Trees initial configuration.



In [ ]:
%%archive_source 03_experiments/model_optimization/extra_trees.py
"""Extra Trees initial configuration."""
from sklearn.ensemble import ExtraTreesClassifier

def build_baseline():
    return ExtraTreesClassifier(random_state=42), False

def build_candidate():
    # Edit n_estimators, max_depth, etc.
    return ExtraTreesClassifier(random_state=42), False


## hgb

원본: [03_experiments/model_optimization/hgb.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/hgb.py)  
Git blob: `a63b379ad3f7a457bae4d5f459a21bff8480d331`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

HistGradientBoosting initial configuration.



In [ ]:
%%archive_source 03_experiments/model_optimization/hgb.py
"""HistGradientBoosting initial configuration."""
from sklearn.ensemble import HistGradientBoostingClassifier

def build_baseline():
    return HistGradientBoostingClassifier(random_state=42), False

def build_candidate():
    # Edit max_iter, learning_rate, max_leaf_nodes, etc.
    return HistGradientBoostingClassifier(random_state=42), False


## knn

원본: [03_experiments/model_optimization/knn.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/knn.py)  
Git blob: `0f2667be1124943ad81751ecd0ced68fd9b76720`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

KNN initial configuration.



In [ ]:
%%archive_source 03_experiments/model_optimization/knn.py
"""KNN initial configuration."""
from sklearn.neighbors import KNeighborsClassifier

def build_baseline():
    return KNeighborsClassifier(), True

def build_candidate():
    # Edit n_neighbors, weights, metric, etc.
    return KNeighborsClassifier(), True


## mlp

원본: [03_experiments/model_optimization/mlp.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/mlp.py)  
Git blob: `8377932f520d1efd9560856d9e3e1f3ce8e68cb8`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

MLP initial configuration.



In [ ]:
%%archive_source 03_experiments/model_optimization/mlp.py
"""MLP initial configuration."""
from sklearn.neural_network import MLPClassifier

def build_baseline():
    return MLPClassifier(random_state=42), True

def build_candidate():
    # Edit hidden_layer_sizes, alpha, max_iter, etc.
    return MLPClassifier(random_state=42), True


## pixel_transform

원본: [03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/pixel_transform.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/pixel_transform.py)  
Git blob: `2878984820965b7c8b1478dc6797069eed41de4b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

Reusable 0-255 -> 0-1 transformer for sklearn pipelines saved by joblib.



In [ ]:
%%archive_source 03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/pixel_transform.py
"""Reusable 0-255 -> 0-1 transformer for sklearn pipelines saved by joblib."""
from sklearn.base import BaseEstimator, TransformerMixin

class PixelScaler255(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X / 255.0


## run

원본: [03_experiments/model_optimization/run.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/run.py)  
Git blob: `53dd28043a7250a1fa0e935e4945d16e42302b98`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

Train fresh baseline/candidate estimators using EXP-유-003's exact split.

No pretrained .joblib loading; official Test and Handmade are not read.

CLI 인자 (정적 추출): `--model`, `--variant`, `--data`, `--output`

In [ ]:
%%archive_source 03_experiments/model_optimization/run.py
#!/usr/bin/env python3
"""Train fresh baseline/candidate estimators using EXP-유-003's exact split.

No pretrained .joblib loading; official Test and Handmade are not read.
"""
from __future__ import annotations
import argparse, csv, importlib, json, platform, time, warnings
from pathlib import Path
import numpy as np
import sklearn
from common import fixed_split, pipeline, scores, DEFAULT_DATA, LABELS

KEYS = ("svc", "hgb", "mlp", "extra_trees", "knn")
BASELINE_ACCURACY = {
 "svc":.9766666666666667, "hgb":.9746354166666666,
 "mlp":.9731770833333333, "extra_trees":.9696875,
 "knn":.9629166666666666,
}

def run_model(key, variant, arrays, outdir):
    xtr,ytr,xva,yva,sha=arrays
    module=importlib.import_module(key)
    estimator,scaled=(module.build_baseline() if variant=="baseline" else module.build_candidate())
    model=pipeline(estimator,scaled)
    t=time.perf_counter()
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter("always")
        model.fit(xtr,ytr)
    fit=time.perf_counter()-t
    t=time.perf_counter()
    pred=model.predict(xva)
    predict=time.perf_counter()-t
    result={
       "key":key,"variant":variant,
       "dataset_sha256":sha,"train_n":len(ytr),"validation_n":len(yva),
       "split":"StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42), first fold",
       "scaling":"pixel/255" if scaled else "raw 0-255",
       "hyperparameters":{str(k):repr(v) for k,v in estimator.get_params().items()},
       "fit_seconds":fit,"predict_seconds":predict,
       "warnings":[f"{w.category.__name__}: {w.message}" for w in ws],
       "python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,
       "historical_baseline_accuracy":BASELINE_ACCURACY[key],
       **scores(yva,pred)
    }
    target=outdir/variant/key
    target.mkdir(parents=True,exist_ok=True)
    (target/"metrics.json").write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding="utf-8")
    with (target/"confusion_matrix.csv").open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.writer(f)
        w.writerow(["true/pred",*LABELS])
        for c,row in zip(LABELS,result["confusion_matrix"]):w.writerow([c,*row])
    with (target/"per_class.csv").open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.writer(f);w.writerow(["class","precision","recall","f1","support"])
        for c,d in result["per_class"].items():w.writerow([c,d["precision"],d["recall"],d["f1"],d["support"]])
    print(f"{key} {variant}: accuracy={result['accuracy']:.6f}, macro_f1={result['macro_f1']:.6f}, historical={BASELINE_ACCURACY[key]:.6f}",flush=True)
    return result

def main():
    p=argparse.ArgumentParser()
    p.add_argument("--model",required=True,choices=[*KEYS,"all"])
    p.add_argument("--variant",choices=["baseline","candidate"],default="baseline")
    p.add_argument("--data",type=Path,default=DEFAULT_DATA)
    p.add_argument("--output",type=Path,default=Path(__file__).resolve().parent/"outputs")
    args=p.parse_args()
    arrays=fixed_split(args.data)
    for key in (KEYS if args.model=="all" else [args.model]):
        run_model(key,args.variant,arrays,args.output)

if __name__=="__main__":
    main()


## run_experiment2

원본: [03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/run_experiment2.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/run_experiment2.py)  
Git blob: `f3895529c8715f1292ce89bbe9a991a25c175524`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

W02 Experiment 2: five Normal (MNIST+EMNIST) baseline models.

Data: final v3 official TRAIN only (96k) => group-aware train 76,800 / val 19,200.
Never loads official X_test/y_test or Handmade for this experiment.
Defaults: sklearn default hyperparameters; only random_state=42 when present.
Scaling: SVC/MLP/KNN pixel/255, tree ensembles unscaled.

Execution from any directory:
  OPENBLAS_NUM_THREADS=2 OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 \
   python run_experiment2.py --prepare
  ... python run_e

CLI 인자 (정적 추출): `--prepare`, `--model`, `--all`, `--timeout`, `--input`, `--no-save-models`

In [ ]:
%%archive_source 03_experiments/W02_2026-11-09_11-15/scripts/experiment2_baseline/run_experiment2.py
#!/usr/bin/env python3
"""W02 Experiment 2: five Normal (MNIST+EMNIST) baseline models.

Data: final v3 official TRAIN only (96k) => group-aware train 76,800 / val 19,200.
Never loads official X_test/y_test or Handmade for this experiment.
Defaults: sklearn default hyperparameters; only random_state=42 when present.
Scaling: SVC/MLP/KNN pixel/255, tree ensembles unscaled.

Execution from any directory:
  OPENBLAS_NUM_THREADS=2 OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 \\
   python run_experiment2.py --prepare
  ... python run_experiment2.py --model mlp
  ... python run_experiment2.py --all --timeout 3600

Each model writes metrics/<key>.json, metrics/<key>_confusion.csv,
metrics/<key>_per_class.csv, metrics/<key>_predictions.npz, models/<key>.joblib.
"""
from __future__ import annotations
import argparse, csv, hashlib, json, os, platform, subprocess, sys, time, warnings
from pathlib import Path
import numpy as np
from sklearn import __version__ as SK_VERSION
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_recall_fscore_support
from joblib import dump
from pixel_transform import PixelScaler255

ROOT=Path(__file__).resolve().parent
SOURCE=Path(__file__).resolve().parents[4] / '02_data' / 'datasets' / 'MNIST_EMNIST_official_split_112000.npz'
EXPECTED_SHA='0e9570d0eb04c0c1da99d043b85bfdab7f3b639eebe4b10335b05b57ad671879'
LABELS='0123456789ABCDEF'
SPECS={
    'extra_trees': ('ExtraTreesClassifier',False,lambda:ExtraTreesClassifier(random_state=42)),
    'mlp': ('MLPClassifier',True,lambda:MLPClassifier(random_state=42)),
    'knn': ('KNeighborsClassifier',True,lambda:KNeighborsClassifier()),
    'svc': ('SVC (RBF)',True,lambda:SVC()),
    'hgb': ('HistGradientBoostingClassifier',False,lambda:HistGradientBoostingClassifier(random_state=42)),
}

def verify_source(path:Path):
    with path.open('rb') as f:sha=hashlib.file_digest(f,'sha256').hexdigest()
    if sha!=EXPECTED_SHA:raise ValueError(f'Unexpected NPZ SHA256: {sha}')
    return sha

def load_training(path:Path):
    sha=verify_source(path)
    with np.load(path,allow_pickle=False) as data:
        x=data['X_train']
        y=data['y_train'].astype(np.int64)
        src=data['train_source_index'].astype(np.int64)
        sp=data['train_source_split'].astype(np.int64)
        aug=data['train_is_augmented']
    assert x.shape==(96000,28,28) and len(y)==96000
    assert len(np.unique(y))==16 and int(aug.sum())==5349
    # Exact source group: class distinguishes MNIST digits / EMNIST letters,
    # split distinguishes original partitions; augmentation source reuses same idx.
    groups=y*1_000_000_000+sp*100_000_000+src
    return sha,x,y,groups,aug

def make_split(path):
    sha,x,y,groups,aug=load_training(path)
    cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
    tr,va=next(cv.split(x,y,groups))
    tr=np.asarray(tr,dtype=np.int32);va=np.asarray(va,dtype=np.int32)
    assert len(tr)==76800 and len(va)==19200
    assert len(set(tr).intersection(va))==0
    assert len(set(groups[tr]).intersection(groups[va]))==0
    assert sorted(np.r_[tr,va].tolist())==list(range(96000))
    (ROOT/'metrics').mkdir(exist_ok=True,parents=True)
    np.savez_compressed(ROOT/'metrics'/'split_indices.npz',train_idx=tr,val_idx=va,train_group_ids=groups[tr],val_group_ids=groups[va])
    meta={
      'study':'W02 Experiment 2 Normal dataset baseline',
      'data_path':str(path),'data_sha256':sha,
      'source_training_n':len(y),'source_official_test_n_NOT_USED':16000,
      'train_n':len(tr),'val_n':len(va),'train_to_validation':'8:2 exact',
      'split_method':'StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42), first fold',
      'split_random_state':42,'train_val_source_group_overlap':0,
      'train_class_counts':{c:int(np.sum(y[tr]==i)) for i,c in enumerate(LABELS)},
      'validation_class_counts':{c:int(np.sum(y[va]==i)) for i,c in enumerate(LABELS)},
      'train_augmented_n':int(aug[tr].sum()),'val_augmented_n':int(aug[va].sum()),
      'feature_shape':[28,28], 'flattened_dim':784,
      'runtime':{'python':platform.python_version(),'sklearn':SK_VERSION,'numpy':np.__version__, 'platform':platform.platform()},
      'evaluation_policy':'Official Test 16,000 and Hand-made 2,560 are NOT used in Experiment 2. Fit only on group-aware Training, evaluate fixed Validation.',
      'historical_note':'A separate interrupted prior preliminary attempt had previously assessed some models on the official Test. Those earlier results are not used here, but the Test is not literally never-viewed in the history of this project.'
    }
    (ROOT/'metrics'/'split_metadata.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
    print(f'SPLIT OK train={len(tr)} val={len(va)} group overlap=0, source_sha256={sha}',flush=True)
    print('class counts:',meta['train_class_counts'],meta['validation_class_counts'],flush=True)

def run_model(key,path,save_models=True):
    if not (ROOT/'metrics'/'split_indices.npz').exists():raise FileNotFoundError('Run --prepare first')
    sha,x,y,groups,aug=load_training(path)
    with np.load(ROOT/'metrics'/'split_indices.npz',allow_pickle=False) as idx:
        tr=idx['train_idx'];va=idx['val_idx']
        stored_train_ids=idx['train_group_ids'];stored_val_ids=idx['val_group_ids']
    assert len(tr)==76800 and len(va)==19200
    assert np.array_equal(groups[tr],stored_train_ids) and np.array_equal(groups[va],stored_val_ids)
    assert not set(groups[tr]).intersection(groups[va])
    Xtr=x[tr].reshape(-1,784).astype(np.float32)
    Xva=x[va].reshape(-1,784).astype(np.float32)
    ytr=y[tr];yva=y[va]
    name,scale,build=SPECS[key]
    impl=build()
    model=Pipeline([('scale_0_1',PixelScaler255()),('model',impl)]) if scale else impl
    print(f'START {key} name={name} train={len(tr)} val={len(va)} scaled={scale}',flush=True)
    t=time.perf_counter()
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter('always')
        model.fit(Xtr,ytr)
    fit=time.perf_counter()-t
    print(f'FIT {key}: {fit:.3f}s',flush=True)
    st=time.perf_counter();yvpred=model.predict(Xva);pred=time.perf_counter()-st
    # Important: same fixed training diagnostic sample, not full training evaluation.
    rng=np.random.default_rng(56)
    sampled=np.concatenate([rng.choice(np.flatnonzero(ytr==i),size=100,replace=False) for i in range(16)])
    st=time.perf_counter();trpred=model.predict(Xtr[sampled]);train_pred=time.perf_counter()-st
    train_probe_acc=accuracy_score(ytr[sampled],trpred)
    cm=confusion_matrix(yva,yvpred,labels=np.arange(16))
    acc=accuracy_score(yva,yvpred)
    f1m=f1_score(yva,yvpred,average='macro',labels=np.arange(16),zero_division=0)
    precis,recall,f1,support=precision_recall_fscore_support(yva,yvpred,labels=np.arange(16),zero_division=0)
    warning_text=[f'{w.category.__name__}: {w.message}' for w in ws]
    niter=getattr(impl,'n_iter_',None)
    if niter is not None:
        niter=np.asarray(niter).tolist()
    result={
        'experiment':'EXP-유-003 / W02 Experiment 2 — Normal baseline',
        'key':key,'model':name,'status':'completed','data_sha256':sha,
        'n_train':len(tr),'n_validation':len(va),'validation_accuracy':float(acc),'validation_macro_f1':float(f1m),
        'training_seconds':float(fit),'validation_inference_seconds':float(pred),
        'validation_ms_per_image':float(pred*1000/len(va)),
        'training_probe_accuracy':float(train_probe_acc),'training_probe_size':1600,
        'training_probe_predict_seconds':float(train_pred),
        'warnings':warning_text,'n_iter':niter,
        'scaled_pixels_0_to_1':scale,'hyperparameters':{str(k):repr(v) for k,v in impl.get_params().items()},
        'model_filename':f'models/{key}.joblib' if save_models else None,
        'class_label_order':LABELS,
        'per_class':{c:{'precision':float(precis[i]),'recall':float(recall[i]),'f1':float(f1[i]),'support':int(support[i])} for i,c in enumerate(LABELS)},
        'confusion_matrix':cm.tolist(),
        'runtime':{'python':platform.python_version(),'scikit-learn':SK_VERSION,'numpy':np.__version__},
    }
    assert sum(cm.diagonal())/len(yva)==acc
    assert abs(np.mean(f1)-f1m)<1e-12
    (ROOT/'metrics').mkdir(exist_ok=True)
    with (ROOT/'metrics'/f'{key}_confusion_matrix.csv').open('w',newline='',encoding='utf-8-sig') as f:
        w=csv.writer(f);w.writerow(['actual\\pred']+list(LABELS));
        for i,c in enumerate(LABELS):w.writerow([c]+cm[i].tolist())
    with (ROOT/'metrics'/f'{key}_per_class.csv').open('w',newline='',encoding='utf-8-sig') as f:
        w=csv.writer(f);w.writerow(['label','precision','recall','f1','support']);
        for c in LABELS:
            a=result['per_class'][c];w.writerow([c,a['precision'],a['recall'],a['f1'],a['support']])
    np.savez_compressed(ROOT/'metrics'/f'{key}_predictions.npz',y_true=yva.astype(np.uint8),y_pred=yvpred.astype(np.uint8),validation_indices=va)
    (ROOT/'metrics'/f'{key}.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
    if save_models:
        (ROOT/'models').mkdir(exist_ok=True)
        dump(model,ROOT/'models'/f'{key}.joblib',compress=3)
    print(f'END {key}: acc={acc:.6f} macro_f1={f1m:.6f} train_probe_acc={train_probe_acc:.6f} training_s={fit:.3f} val_predict_s={pred:.3f} warnings={len(warning_text)}',flush=True)

def aggregate():
    import csv
    rows=[]
    for key,(name,scale,build) in SPECS.items():
        p=ROOT/'metrics'/f'{key}.json'
        if p.exists():
            d=json.loads(p.read_text(encoding='utf-8'))
            rows.append({
                'model':name,'status':d['status'],'train_n':d['n_train'],'val_n':d['n_validation'],
                'train_sample_accuracy':d['training_probe_accuracy'],'validation_accuracy':d['validation_accuracy'],
                'macro_f1':d['validation_macro_f1'],'fit_seconds':d['training_seconds'],
                'predict_seconds':d['validation_inference_seconds'],'predict_ms_per_image':d['validation_ms_per_image'],
                'scaling':'pixel / 255' if d['scaled_pixels_0_to_1'] else 'raw pixels 0-255',
                'n_iter':d['n_iter'],'warnings':' | '.join(d['warnings'])
            })
        else:rows.append({'model':name,'status':'not_completed'})
    (ROOT/'metrics').mkdir(exist_ok=True)
    cols=['model','status','train_n','val_n','train_sample_accuracy','validation_accuracy','macro_f1','fit_seconds','predict_seconds','predict_ms_per_image','scaling','n_iter','warnings']
    with (ROOT/'metrics'/'comparison_5models.csv').open('w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=cols);w.writeheader()
        for row in rows:w.writerow(row)
    (ROOT/'metrics'/'comparison_5models.json').write_text(json.dumps(rows,indent=2,ensure_ascii=False),encoding='utf-8')
    print('AGGREGATED', [(r['model'],r['status']) for r in rows],flush=True)

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--prepare',action='store_true')
    ap.add_argument('--model',choices=SPECS)
    ap.add_argument('--all',action='store_true')
    ap.add_argument('--timeout',type=int,default=3600)
    ap.add_argument('--input',type=Path,default=SOURCE)
    ap.add_argument('--no-save-models',action='store_true')
    a=ap.parse_args()
    if a.prepare:make_split(a.input)
    elif a.model:run_model(a.model,a.input,not a.no_save_models)
    elif a.all:
        if not (ROOT/'metrics'/'split_indices.npz').exists():make_split(a.input)
        env={**os.environ,'OPENBLAS_NUM_THREADS':'2','OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2','NUMEXPR_NUM_THREADS':'2'}
        for key in SPECS:
            if (ROOT/'metrics'/f'{key}.json').exists():continue
            print(f'==== START {key} ==== ',flush=True)
            st=time.perf_counter()
            cmd=['timeout','--signal=TERM','--kill-after=5s',f'{a.timeout}s',sys.executable,str(Path(__file__).resolve()),'--model',key,'--input',str(a.input)]
            if a.no_save_models:cmd.append('--no-save-models')
            p=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=env,cwd=ROOT)
            print(p.stdout[-2500:],flush=True)
            if p.returncode:
                reason={'key':key,'model':SPECS[key][0],'status':'timeout' if p.returncode==124 else 'failed','exit_code':p.returncode,'wall_elapsed_seconds':time.perf_counter()-st,'log_tail':p.stdout[-3000:]}
                (ROOT/'metrics'/f'{key}_failure.json').write_text(json.dumps(reason,ensure_ascii=False,indent=2))
            aggregate()
        aggregate()
    else:ap.error('Choose --prepare, --model or --all')
if __name__=='__main__':main()

## svc

원본: [03_experiments/model_optimization/svc.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/model_optimization/svc.py)  
Git blob: `c4af494a2b9d6e2367c3d93d1b9fccc88ac7b6c3`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

SVC initial configuration: keep baseline; edit candidate.



In [ ]:
%%archive_source 03_experiments/model_optimization/svc.py
"""SVC initial configuration: keep baseline; edit candidate."""
from sklearn.svm import SVC

def build_baseline():
    return SVC(), True

def build_candidate():
    # Edit C/gamma/kernel, or return scaling=False for experiments.
    return SVC(), True


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")